# EZ-Pair Graph Demo Notebook

**Scalable unified-axis visualization for summarizing large-scale paired data**

This notebook demonstrates how to use the `ez-pair-graph` Python package for visualizing paired data. EZ-Pair Graph provides four complementary visualization types:

1. **Slope Graph** — Individual paired observations as connecting lines
2. **Trapezoid Plot** — Quartile-based summary bands for ascending/descending groups
3. **Clustered Line Plot** — Cluster-level summaries with half-boxplots
4. **Parallel Arrow Plot** — Direction and magnitude of cluster-level changes

**Reference:** Ezoe, A., Seki, M., & Mochida, K. (2026). EZ-Pair Graph: Scalable unified-axis visualization for summarizing large-scale paired data. *Bioinformatics Advances*.

**License:** Academic and non-commercial research use only (RIKEN). The technology is currently under patent application.

In [ ]:
# Install ez-pair-graph (required for Google Colab)
!pip install git+https://github.com/010049nn/EZ_pair_graph.git -q

## 1. Installation

```bash
pip install git+https://github.com/010049nn/EZ_pair_graph.git
```

Or install from source:
```bash
git clone https://github.com/010049nn/EZ_pair_graph.git
cd EZ_pair_graph
pip install -e .
```

In [ ]:
import ez_pair_graph as ezpg
import numpy as np
import pandas as pd
from IPython.display import Image, display

print(f"ez-pair-graph version: {ezpg.__version__}")

## 2. Quick Start with Synthetic Data

### 2.1 Generate paired data with a known shift

We create synthetic paired observations where group B values are shifted upward relative to group A, simulating a treatment effect.

In [ ]:
# Generate synthetic paired data
np.random.seed(42)
n = 300

# Group A: normal distribution (mean=50, SD=10)
group_a = np.random.normal(50, 10, n)

# Group B: shifted by +5 with correlated noise
group_b = group_a + np.random.normal(5, 8, n)

print(f"Sample size: {n}")
print(f"Group A: mean={group_a.mean():.2f}, SD={group_a.std():.2f}")
print(f"Group B: mean={group_b.mean():.2f}, SD={group_b.std():.2f}")
print(f"Mean difference (B - A): {(group_b - group_a).mean():.2f}")

### 2.2 Generate all four plots with `plot_array()`

In [ ]:
# Generate all four EZ-Pair Graph visualizations
figures = ezpg.plot_array(
    group_a, group_b,
    output_dir="output_demo",
    format="png"
)

print(f"Generated {len(figures)} plots:")
for name, path in figures.items():
    print(f"  - {name}: {path}")

### 2.3 Display plots inline

In [ ]:
# Slope Graph
print("=== Slope Graph ===")
display(Image(filename=figures["slopegraph"]))

In [ ]:
# Trapezoid Plot
print("=== Trapezoid Plot ===")
display(Image(filename=figures["trapezoid"]))

In [ ]:
# Clustered Line Plot (with half-boxplots)
print("=== Clustered Line Plot ===")
display(Image(filename=figures["clustered_line"]))

In [ ]:
# Parallel Arrow Plot
print("=== Parallel Arrow Plot ===")
display(Image(filename=figures["parallel_arrow"]))

## 3. Working with DataFrames

`plot_dataframe()` accepts a pandas DataFrame directly.

In [ ]:
# Create a DataFrame
df = pd.DataFrame({
    "before_treatment": group_a,
    "after_treatment": group_b,
    "sample_id": [f"S{i+1}" for i in range(n)]
})

print(df.head(10))

In [ ]:
# Generate specific plots from DataFrame
figures_df = ezpg.plot_dataframe(
    df,
    x_col="before_treatment",
    y_col="after_treatment",
    output_dir="output_dataframe",
    format="png",
    plots=["trapezoid", "clustered_line"]
)

print(f"Generated: {list(figures_df.keys())}")
for name, path in figures_df.items():
    display(Image(filename=path))

## 4. File-Based Workflow

The `plot()` function reads directly from CSV/TSV files, suitable for integration into analysis pipelines.

In [ ]:
# Save test data to CSV
df[["before_treatment", "after_treatment"]].to_csv("test_data.csv", index=False)

# Generate plots from file
figures_file = ezpg.plot(
    "test_data.csv",
    output_dir="output_from_file",
    format="png",
    plots=["slopegraph", "trapezoid"]
)

print(f"Generated: {list(figures_file.keys())}")
print("Output saved to: output_from_file/")
for name, path in figures_file.items():
    display(Image(filename=path))

## 5. Clustering Methods

### Hierarchical Clustering (default)
Uses Ward's method with automatic k selection via the elbow method.

### HDBSCAN Clustering
Density-based clustering that automatically determines the number of clusters and can identify noise points.

In [ ]:
# Hierarchical clustering (default)
figures_hier = ezpg.plot_array(
    group_a, group_b,
    output_dir="output_hierarchical",
    format="png",
    method="hierarchical",
    max_k=7,
    plots=["clustered_line"]
)

print("=== Hierarchical Clustering ===")
display(Image(filename=figures_hier["clustered_line"]))

In [ ]:
# HDBSCAN clustering
figures_hdb = ezpg.plot_array(
    group_a, group_b,
    output_dir="output_hdbscan",
    format="png",
    method="hdbscan",
    min_cluster_size=10,
    plots=["clustered_line"]
)

print("=== HDBSCAN Clustering ===")
display(Image(filename=figures_hdb["clustered_line"]))

## 6. Additional Options

### Log2 transformation
Useful for gene expression data or other log-scale measurements.

### Outlier filtering
Removes outlier display in plots.

In [ ]:
# Log2 transformation example (simulating expression data)
np.random.seed(99)
expr_a = np.random.lognormal(mean=5, sigma=1, size=200)
expr_b = expr_a * np.random.lognormal(mean=0.3, sigma=0.5, size=200)

figures_log2 = ezpg.plot_array(
    expr_a, expr_b,
    output_dir="output_log2",
    format="png",
    log2=True,
    plots=["slopegraph", "trapezoid"]
)

print("=== Log2 Transformed Plots ===")
for name, path in figures_log2.items():
    print(f"\n--- {name} ---")
    display(Image(filename=path))

## 7. Large-Scale Data (n > 1,000)

EZ-Pair Graph is designed to handle large datasets. The trapezoid plot and clustered line plot provide clear summaries even with thousands of data points.

In [ ]:
# Large-scale example (n=5000)
np.random.seed(7)
n_large = 5000
x_large = np.random.normal(100, 20, n_large)
y_large = x_large + np.random.normal(3, 15, n_large)

figures_large = ezpg.plot_array(
    x_large, y_large,
    output_dir="output_large",
    format="png"
)

print(f"Generated all 4 plots for n={n_large}")
for name, path in figures_large.items():
    print(f"\n=== {name} (n={n_large}) ===")
    display(Image(filename=path))

## 8. Command-Line Interface

EZ-Pair Graph also provides a CLI for batch processing:

```bash
# Basic usage
ez-pair-graph data.txt

# PNG output with HDBSCAN clustering
ez-pair-graph data.txt --format png --method hdbscan

# Specific plots only
ez-pair-graph data.txt --plots trapezoid clustered_line

# With log2 transformation and outlier removal
ez-pair-graph data.txt --log2 --no-outliers

# Custom output directory
ez-pair-graph data.txt --output-dir my_results --output-prefix experiment1
```

## Summary

| Function | Input | Description |
|----------|-------|-------------|
| `ezpg.plot(file)` | File path | Full pipeline from file |
| `ezpg.plot_array(x, y)` | NumPy arrays | Full pipeline from arrays |
| `ezpg.plot_dataframe(df, x_col, y_col)` | DataFrame | Full pipeline from DataFrame |

For more details, see the [GitHub repository](https://github.com/010049nn/EZ_pair_graph) and the accompanying paper.

---
*Copyright (c) 2025. RIKEN All rights reserved. Academic and non-commercial research use only.*